In [1]:

"""
Enhanced Multi-modal Building Geometry Extraction Pipeline

3 modes:
  Mode 1: Single Model + Single Sample (Baseline)
  Mode 2: Single Model + Multi Sample (Self-consistency)
  Mode 3: Multi Model + Single Sample (Cross-consistency)  
"""

import base64
import json
import re
import os
import pprint
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Union
from collections import Counter
from PIL import Image
from io import BytesIO

# API Clients
from google import genai
from google.genai import types
from openai import OpenAI
from anthropic import Anthropic

# =========================
# Configs:
# • "baseline"           → single model, single sample; fast and cheap
# • "self_consistency"   → single model, multiple sample; relatively economical and accurate
# • "cross_consistency"  → all providers run together, each sampled multiple times, then aggregate by consensus; highly robust and accurate
# =========================

picture_path = "./Standard_Test"  # standard set and boundary exploration sets
RUN_MODE = "baseline"   # "baseline", "self_consistency", "cross_consistency"
candidate = "gemini"            # only for baseline/self_consistency: "gemini", "qwen", "openai", "claude"

if RUN_MODE == "cross_consistency":
    output_py = f"user_prompts_3_{RUN_MODE}.py"
else:
    output_py = f"user_prompts_3_{RUN_MODE}_{candidate}.py"
    
MODE_CONFIGS = {
    "baseline": {
        "description": "Single Model + Multi Sample Baseline",
        "providers": ["gemini"],
        "samples_per_model": 3,
        "temperature": 0,
        "cross_consistency": False,
        "self_consistency": False
    },
    "self_consistency": {
        "description": "Sequential Multi-Model Multi Sample",
        "providers": ["gemini", "openai", "qwen", "claude"],
        "samples_per_model": 3,
        "temperature": 0,
        "cross_consistency": False,
        "self_consistency": True
    },
    "cross_consistency": {
        "description": "Parallel Multi-Model Multi Sample Consensus",
        "providers": ["gemini", "openai", "qwen", "claude"],
        "samples_per_model": 3,
        "temperature": 0,
        "cross_consistency": True,
        "self_consistency": True
    }
}

# =========================
# strategies
# =========================
STRATEGY_CONFIG = {
    "self_consistency": {
        "enabled": True,
        "num_samples": 3,
        "temperature": 0,
        "aggregation": "consensus"
    },
    "reflective_verification": {
        "enabled": True,
        "max_reflection_rounds": 1,
        "rules": {
            "min_dimension_m": 0.01,
            "max_dimension_m": 10000,
            "require_positive": True,
            "shape_constraints": {
                "Rectangle": ["rectangle_overall_length_m", "rectangle_overall_width_m"],
                "L-shaped": ["l_horizontal_length_m", "l_horizontal_width_m", "l_vertical_length_m", "l_vertical_width_m"],
                "U-shaped": ["u_main_length_m", "u_main_width_m", "u_left_length_m", "u_left_width_m", "u_right_length_m", "u_right_width_m"],
                "T-shaped": ["t_horizontal_length_m", "t_horizontal_width_m", "t_vertical_length_m", "t_vertical_width_m", "t_edge_offset_m"],
                "Courtyard": ["courtyard_horizontal_length_m", "courtyard_horizontal_width_m", "courtyard_vertical_length_m", "courtyard_vertical_width_m"]
            }
        }
    },
    "cross_consistency": {
        "enabled": True,
        "providers": ["gemini", "openai", "qwen", "claude"],
        "require_full_confidence": True
    }
}

# =========================
# api key
# =========================

gang_api_key1 = "" # put your api keys
gang_api_key2 = ""
gang_api_key3 = ""
gang_api_key4 = ""
gang_api_key5 = ""
gang_api_keyc = ""

claude_client = Anthropic(
    api_key="" # your api
)

client = genai.Client(api_key=gang_api_keyc)  # Gemini

openai_client = OpenAI( # GPT
    api_key="" # your api
)

qwen_client = OpenAI( # Qwen
    api_key="", # your api
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

# =========================
# functions
# =========================
def encode_image(image_path: str) -> str:
    """Base64 encoded"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

    
def parse_geometry_output(text: str) -> Optional[Dict]:

    if not text or "##final answer##" not in text: # tag to abstract
        return None
    
    result = {"raw": text, "shape": None, "dimensions": {}, "_source_model": None}
    
    shape_match = re.search(r'Building shape:\s*([A-Za-z]+(?:[-\s][A-Za-z]+)?)', text, re.I)
    if shape_match:
        raw_shape = shape_match.group(1).strip().lower()
        shape_map = {
            "rectangular": "Rectangle", "rectangle": "Rectangle",
            "l-shaped": "L-shaped", "l shaped": "L-shaped", "lshaped": "L-shaped",
            "u-shaped": "U-shaped", "u shaped": "U-shaped", "ushaped": "U-shaped",
            "t-shaped": "T-shaped", "t shaped": "T-shaped", "tshaped": "T-shaped",
            "courtyard": "Courtyard"
        }
        result["shape"] = shape_map.get(raw_shape, raw_shape.replace(" ", "-").title())
    
    dim_pattern = r'(?:^|\n)\s*(?:[-•]\s*)?([a-zA-Z_][a-zA-Z0-9_]*_m)\s*:\s*([-+]?\d+\.?\d*(?:[eE][-+]?\d+)?)'
    matches = re.findall(dim_pattern, text, re.MULTILINE)
    
    for key, val in matches:
        clean_key = key.strip().lower()  
        try:
            result["dimensions"][clean_key] = float(val)
        except ValueError:
            continue  
    
    return result if result["shape"] else None


################################
# verfity geometry #
################################
def verify_geometry(parsed: Dict, rules: Dict) -> Tuple[bool, List[str]]:

    errors = []
    dims = parsed.get("dimensions", {})
    shape = parsed.get("shape")

    if rules.get("require_positive", True):
        for k, v in dims.items():
            if v <= 0:
                errors.append(f"Dimension '{k}' = {v} is not positive")

    min_m = rules.get("min_dimension_m", 1)
    max_m = rules.get("max_dimension_m", 1000)
    for k, v in dims.items():
        if not (min_m <= v <= max_m):
            errors.append(f"Dimension '{k}' = {v}m out of range [{min_m}, {max_m}]")

    # shape checking
    shape_constraints = rules.get("shape_constraints", {})
    if shape and shape in shape_constraints:
        required_keys = shape_constraints[shape]
        for key in required_keys:
            if key not in dims:
                errors.append(f"Shape '{shape}' requires dimension '{key}'")

    # -------------------------
    # L-shaped
    # -------------------------
    if shape == "L-shaped":
        h_len = dims.get("l_horizontal_length_m")
        h_wid = dims.get("l_horizontal_width_m")
        v_len = dims.get("l_vertical_length_m")
        v_wid = dims.get("l_vertical_width_m")

        # l_horizontal_length_m > l_vertical_length_m
        if h_len is not None and v_len is not None:
            if h_len <= v_len:
                errors.append(
                    f"For shape 'L-shaped', 'l_horizontal_length_m' ({h_len}) "
                    f"must be greater than 'l_vertical_length_m' ({v_len})"
                )

    # -------------------------
    # T-shaped
    # -------------------------
    elif shape == "T-shaped":
        h_len = dims.get("t_horizontal_length_m")
        h_wid = dims.get("t_horizontal_width_m")
        v_len = dims.get("t_vertical_length_m")
        v_wid = dims.get("t_vertical_width_m")
        edge_offset = dims.get("t_edge_offset_m")

        # t_horizontal_length_m > t_vertical_length_m
        if h_len is not None and v_len is not None:
            if h_len <= v_len:
                errors.append(
                    f"For shape 'T-shaped', 't_horizontal_length_m' ({h_len}) "
                    f"must be greater than 't_vertical_length_m' ({v_len})"
                )

        # 0 < t_edge_offset_m < t_horizontal_length_m
        if edge_offset is not None:
            if edge_offset <= 0:
                errors.append(
                    f"For shape 'T-shaped', 't_edge_offset_m' ({edge_offset}) must be greater than 0"
                )
            if h_len is not None and edge_offset >= h_len:
                errors.append(
                    f"For shape 'T-shaped', 't_edge_offset_m' ({edge_offset}) "
                    f"must be smaller than 't_horizontal_length_m' ({h_len})"
                )

    # -------------------------
    # U-shaped
    # -------------------------
    elif shape == "U-shaped":
        main_len = dims.get("u_main_length_m")
        main_wid = dims.get("u_main_width_m")
        left_len = dims.get("u_left_length_m")
        left_wid = dims.get("u_left_width_m")
        right_len = dims.get("u_right_length_m")
        right_wid = dims.get("u_right_width_m")

        # u_main_length_m > u_left_length_m
        if main_len is not None and left_len is not None:
            if main_len <= left_len:
                errors.append(
                    f"For shape 'U-shaped', 'u_main_length_m' ({main_len}) "
                    f"must be greater than 'u_left_length_m' ({left_len})"
                )

        # u_main_length_m > u_right_length_m
        if main_len is not None and right_len is not None:
            if main_len <= right_len:
                errors.append(
                    f"For shape 'U-shaped', 'u_main_length_m' ({main_len}) "
                    f"must be greater than 'u_right_length_m' ({right_len})"
                )

    # -------------------------
    # Courtyard
    # -------------------------
    elif shape == "Courtyard":
        h_len = dims.get("courtyard_horizontal_length_m")
        h_wid = dims.get("courtyard_horizontal_width_m")
        v_len = dims.get("courtyard_vertical_length_m")
        v_wid = dims.get("courtyard_vertical_width_m")

        # horizontal_length > vertical_lentgh
        # horizontal_width < vertical_width
        if h_len is not None and v_len is not None:
            if h_len <= v_len:
                errors.append(
                    f"For shape 'Courtyard', 'courtyard_horizontal_length_m' ({h_len}) "
                    f"must be greater than 'courtyard_vertical_length_m' ({v_len})"
                )

        if h_wid is not None and v_wid is not None:
            if h_wid >= v_wid:
                errors.append(
                    f"For shape 'Courtyard', 'courtyard_horizontal_width_m' ({h_wid}) "
                    f"must be smaller than 'courtyard_vertical_width_m' ({v_wid})"
                )

        if h_len is not None and h_wid is not None:
            if h_len <= h_wid:
                errors.append(
                    f"For shape 'Courtyard', 'courtyard_horizontal_length_m' ({h_len}) "
                    f"must be greater than 'courtyard_horizontal_width_m' ({h_wid})"
                )

        if v_len is not None and v_wid is not None:
            if v_len >= v_wid:
                errors.append(
                    f"For shape 'Courtyard', 'courtyard_vertical_length_m' ({v_len}) "
                    f"must be smaller than 'courtyard_vertical_width_m' ({v_wid})"
                )

    return len(errors) == 0, errors

def robust_mode(values, round_ndigits=3):
    """
    calculate mode value for candidates
    """
    if not values:
        return None
    
    rounded = [round(v, round_ndigits) for v in values]
    
    counter = Counter(rounded)
    most_common = counter.most_common()
    
    max_count = most_common[0][1]
    modes = [v for v, c in most_common if c == max_count]
    
    # 你可以改策略：
    # return float(np.mean(modes))  # change the mode strategy to mean
    return modes[0]             # if multi modes, select the first one 

def aggregate_consensus(results: List[Dict], config: Dict) -> Optional[Dict]:
    """consensus aggregation"""
    
    if not results:
        return None

    # level 1: shape consensus
    shapes = [r["shape"] for r in results if r and r.get("shape")]
    if not shapes:
        return None
    
    shape_counter = Counter(shapes)
    shape_consensus, shape_count = shape_counter.most_common(1)[0]

    shape_consensus_results = [
        r for r in results if r and r.get("shape") == shape_consensus
    ]

    all_dims = [
        r["dimensions"] for r in shape_consensus_results if r and r.get("dimensions")
    ]

    if not all_dims:
        return {
            "shape": shape_consensus,
            "dimensions": {},
            "confidence": 0.0,
            "total_source_count": len(results),
            "shape_consensus_source_count": len(shape_consensus_results),
        }

    # level 2: dimension consensus (use mode aggregation)
    all_keys = set(k for d in all_dims for k in d.keys())
    aggregated_dims = {}
    dim_confidences = {}

    for key in all_keys:
        values = [
            d[key] for d in all_dims
            if key in d and isinstance(d[key], (int, float))
        ]
        
        if values:
            mode_value = robust_mode(values)
            aggregated_dims[key] = mode_value

            rounded_values = [round(v, 3) for v in values]
            rounded_mode = round(mode_value, 3)
            match_count = sum(1 for v in rounded_values if v == rounded_mode)

            dim_confidences[key] = match_count / len(values)

    # total confidence value
    confidence = sum(dim_confidences.values()) / len(dim_confidences) if dim_confidences else 0.0

    return {
        "shape": shape_consensus,
        "dimensions": aggregated_dims,
        "confidence": confidence,
        "dimension_confidences": dim_confidences,
        "total_source_count": len(results),
        "shape_consensus_source_count": len(shape_consensus_results),
    }

def format_output_from_parsed(parsed: Dict, user_prompt: str = "") -> str:

    shape = parsed.get("shape", "Unknown")
    dims = parsed.get("dimensions", {})
    conf = parsed.get("confidence", 1.0)
    
    src_cnt = parsed.get("total_source_count", parsed.get("source_count", 1))
    
    lines = [
        "##final answer##",
        f"1. Building shape: {shape}",
        "2. Building size:"
    ]
    
    if shape == "Rectangle":

        length_val = dims.get('rectangle_overall_length_m', 
                     dims.get('overall_building_length', 
                     dims.get('length', 'N/A')))
        width_val = dims.get('rectangle_overall_width_m', 
                    dims.get('overall_building_width', 
                    dims.get('width', 'N/A')))
        lines.append(f"   Overall building length: {length_val}m")
        lines.append(f"   Overall building width: {width_val}m")
    else:
        for k, v in dims.items():
            lines.append(f"   {k}: {v}m")

    if user_prompt.strip():
        lines.append(f"\n{user_prompt}")
    
    return "\n".join(lines)

def run_single_provider_pipeline(
    provider: str,
    image_path: Path,
    prompt: str,
    samples_per_model: int,
    temperature: float,
    default_provider: str
) -> Tuple[List[str], List[Dict]]:
    """
    for single provider：
    1) run multi samples
    2) parse
    3) reflective verification
    4) return candidate_texts, candidate_parsed
    """
    raw_records = []

    for s_idx in range(samples_per_model):
        label = f"{provider}" if samples_per_model == 1 else f"{provider}#{s_idx+1}"
        print(f"  🎲 Calling {label} (temp={temperature})...")

        try:
            text = call_model_single(
                provider=provider,
                image_path=image_path,
                prompt=prompt,
                temperature=temperature,
                stream=False
            )

            parsed = parse_geometry_output(text)
            if parsed:
                parsed["_source_model"] = provider
                parsed["_sample_idx"] = s_idx

                print(f"    ✓ Parsed: shape={parsed['shape']}, dims={len(parsed['dimensions'])}")
                pprint.pprint(parsed["dimensions"])
            else:
                print(f"    ⚠ Parse failed for {label}")
                print(f"    🤖 Full Model Raw Output:")
                print(f"    {'─'*40}")
                print(text)
                print(f"    {'─'*40}")

            raw_records.append({
                "text": text,
                "parsed": parsed,
                "provider": provider,
                "sample_idx": s_idx
            })

        except Exception as e:
            print(f"    ✗ {label} error: {e}")
            continue

    # Reflective Verification
    if STRATEGY_CONFIG["reflective_verification"]["enabled"] and raw_records:
        rules = STRATEGY_CONFIG["reflective_verification"]["rules"]
        max_rounds = STRATEGY_CONFIG["reflective_verification"]["max_reflection_rounds"]

        verified_records = []

        for rec in raw_records:
            raw_text = rec["text"]
            parsed = rec["parsed"]
            provider = rec["provider"]
            sample_idx = rec["sample_idx"]

            if parsed:
                is_valid, errors = verify_geometry(parsed, rules)
                if is_valid:
                    print(f"  ✓ [{provider}] Sample passed verification")
                    verified_records.append(rec)
                else:
                    print(f"  ⚠ [{provider}] Sample failed verification: {errors[:2]}")
                    if max_rounds > 0:
                        corrected = reflect_and_retry(
                            provider=provider,
                            image_path=image_path, # drawings
                            original_prompt=prompt, # picture_system_prompt
                            errors=errors, # error information for extra prompt
                            max_rounds=max_rounds,
                            temperature=0
                        )
                        new_parsed = parse_geometry_output(corrected)
                        if new_parsed:
                            new_parsed["_source_model"] = provider
                            new_parsed["_sample_idx"] = sample_idx
                            new_parsed["_reflected"] = True
                        verified_records.append({
                            "text": corrected,
                            "parsed": new_parsed,
                            "provider": provider,
                            "sample_idx": sample_idx
                        })
                    else:
                        verified_records.append(rec)
            else:
                print(f"  ✗ Sample parse failed, keeping original")
                verified_records.append(rec)

        raw_records = verified_records

    candidate_texts = [r["text"] for r in raw_records]
    candidate_parsed = [r["parsed"] for r in raw_records if r["parsed"]]

    return candidate_texts, candidate_parsed

def finalize_from_candidates(candidate_texts: List[str], candidate_parsed: List[Dict], user_text_prompt: str) -> Tuple[str, Optional[Dict]]:
    """
    final parsed result
    """
    valid_parsed = [p for p in candidate_parsed if p and p.get("shape")]

    if valid_parsed and len(valid_parsed) > 1:
        print(f"  📊 Aggregating {len(valid_parsed)} valid results from {len(set(p.get('_source_model') for p in valid_parsed))} model(s)...")
        final_parsed = aggregate_consensus(valid_parsed, STRATEGY_CONFIG)
        print(
            f"  📋 Aggregation Params: "
            f"shape={final_parsed.get('shape')}, "
            f"confidence={final_parsed.get('confidence'):.2%}, "
            f"total_source_count={final_parsed.get('total_source_count')}, "
            f"shape_consensus_source_count={final_parsed.get('shape_consensus_source_count')}"
        )
        final_text = format_output_from_parsed(final_parsed, user_text_prompt)
        return final_text, final_parsed

    elif valid_parsed:
        final_parsed = valid_parsed[0]
        if "confidence" not in final_parsed:
            final_parsed["confidence"] = 1.0
        final_text = format_output_from_parsed(final_parsed, user_text_prompt)
        return final_text, final_parsed

    elif candidate_texts:
        for t in candidate_texts:
            if "##final answer##" in t:
                final_text = t + (f"\n{user_text_prompt}" if user_text_prompt.strip() else "")
                return final_text, None
        return candidate_texts[0], None

    else:
        return "", None
    
def call_model_single(
    provider: str,
    image_path: Path,
    prompt: str,
    temperature: float = 0.0, 
    stream: bool = False
) -> str:
    """Single run, Gemini / OpenAI / Qwen"""
    if provider.lower() == "openai":
        base64_image = encode_image(str(image_path))
        response = openai_client.responses.create(
            model="gpt-5.4-2026-03-05",
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{base64_image}"}
                ]
            }],
            temperature=temperature
        )
        return (response.output_text or "").strip()
    
    elif provider.lower() == "qwen":
        image_base64 = encode_image(str(image_path))
        if stream:
            completion = qwen_client.chat.completions.create(
                model="qwen3.5-plus",
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
                        {"type": "text", "text": prompt}
                    ]
                }],
                stream=True,
                temperature=temperature,
                extra_body={"enable_thinking": False, "thinking_budget": 500}
            )
            text_parts = []
            for chunk in completion:
                try:
                    delta = chunk.choices[0].delta
                    if delta and getattr(delta, "content", None):
                        text_parts.append(delta.content)
                except Exception:
                    pass
            return "".join(text_parts).strip()
        else:
            completion = qwen_client.chat.completions.create(
                model="qwen3.5-plus", # try other models, if you can
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
                        {"type": "text", "text": prompt}
                    ]
                }],
                temperature=temperature,
                extra_body={"enable_thinking": False}
            )
            return completion.choices[0].message.content.strip()
        
    elif provider.lower() == "claude":
        image_base64 = encode_image(str(image_path))
        suffix = image_path.suffix.lower()
        media_type_map = {
            ".jpg": "image/jpeg",
            ".jpeg": "image/jpeg",
            ".png": "image/png",
            ".webp": "image/webp",
            ".gif": "image/gif",
        }
        media_type = media_type_map.get(suffix, "image/png")

        response = claude_client.messages.create(
            model="claude-opus-4-6",
            max_tokens=2000, # necessary to set, 2k is enough
            temperature=temperature,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": media_type,
                                "data": image_base64
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ]
        )

        text_parts = []
        for block in response.content:
            if getattr(block, "type", None) == "text":
                text_parts.append(block.text)
        return "\n".join(text_parts).strip()
    
    else:  # gemini defaulted, best performance
        image = Image.open(image_path)
        response = client.models.generate_content(
            model="gemini-3-pro-image-preview", # gemini-3.1-pro-preview
            contents=[prompt, image],
        )
        temperature=temperature,
        text_parts = []
        for part in response.candidates[0].content.parts:
            if part.text is not None:
                text_parts.append(part.text)
        return "\n".join(text_parts).strip()


def reflect_and_retry(
    provider: str,
    image_path: Path,
    original_prompt: str,
    errors: List[str],
    max_rounds: int = 1,
    temperature: float = 0
) -> str:
    """verification and revision"""
    reflection_prefix = f"""
⚠️ VERIFICATION FAILED - Please revise your previous answer.

Issues detected:
{chr(10).join(f'  • {e}' for e in errors)}

Requirements for revision:
1. All dimensions must be positive numbers in meters (m)
2. Ensure shape classification matches the actual footprint geometry
3. For complex shapes (L/U/T/Courtyard), verify ALL required segment dimensions are provided
4. Output STRICTLY in the ##final answer## format with no extra explanations

Your revised answer:"""
    
    revised_prompt = original_prompt + reflection_prefix
    
    for round_idx in range(max_rounds):
        try:
            print(f"    🔄 Reflection round {round_idx+1}/{max_rounds} ({provider})...")
            text = call_model_single(
                provider=provider,
                image_path=image_path,
                prompt=revised_prompt,
                temperature=temperature,
                stream=False
            )
            
            parsed = parse_geometry_output(text)
            if parsed:
                rules = STRATEGY_CONFIG["reflective_verification"]["rules"]
                is_valid, new_errors = verify_geometry(parsed, rules)
                if is_valid:
                    print(f"    ✓ Reflection succeeded ({provider})")
                    return text
                else:
                    print(f"    ⚠ Still invalid ({provider}): {new_errors[:2]}")
                    errors = new_errors
            else:
                print(f"    ⚠ Parse failed ({provider}), retrying...")
                
        except Exception as e:
            print(f"    ✗ Reflection error ({provider}): {e}")
            continue
    
    print(f"    ⚠ Reflection exhausted ({provider}), returning last result")
    return text


# =========================
# 📝 user Prompt
# you can input your requirement, besides the building geometry.
# you also can select not to input.
# =========================
# user_text_prompt = """It is a 9 stories building, with heat pump system."""
user_text_prompt = """"""

picture_system_prompt = """
Please analyze the provided architectural floor plan image and answer the following:
Identify the overall building footprint shape.
Estimate the approximate dimensions of each visible segment of the footprint based on the drawing scale or visual proportions.
Length always means a horizontal (X-direction) measurement.
Width always means a vertical (Y-direction) measurement.
Do not classify dimensions based on which side is longer or shorter.
Report all estimated dimensions in meters.

Finally answer the output format, no explanations:
##final answer##
1. Building shape: Possible values: Rectangle, L-shaped, U-shaped, T-shaped, Courtyard.
2. Building size (meters):

For rectangular building, building size including:
- rectangle_overall_length_m (Overall building length)
- rectangle_overall_width_m (Overall building width)

For L-shaped building, building size including:
- l_horizontal_length_m (Horizontal segment length)
- l_horizontal_width_m (Horizontal segment width)
- l_vertical_length_m (Vertical segment length)
- l_vertical_width_m (Vertical segment width)
Important: Length always means a horizontal (X-direction) measurement.
Width always means a vertical (Y-direction) measurement.
These labels do not depend on which side is visually longer or shorter.

For U-shaped building, building size including:
- u_main_length_m (Main segment length)
- u_main_width_m (Main segment width)
- u_left_length_m (Left segment length)
- u_left_width_m (Left segment width)
- u_right_length_m (Right segment length)
- u_right_width_m (Right segment width)
Important: Length always means a horizontal (X-direction) measurement.
Width always means a vertical (Y-direction) measurement.
These labels do not depend on which side is visually longer or shorter.

For T-shaped building, building size including:
- t_horizontal_length_m (Horizontal segment length)
- t_horizontal_width_m (Horizontal segment width)
- t_vertical_length_m (Vertical segment length)
- t_vertical_width_m (Vertical segment width)
- t_edge_offset_m (The distance from the vertical segment to the edge of the horizontal segment)
Important: Length always means a horizontal (X-direction) measurement.
Width always means a vertical (Y-direction) measurement.
These labels do not depend on which side is visually longer or shorter.

For courtyard building, building size including:
- courtyard_horizontal_length_m (Two horizontal segments (have two, always with same size) length)
- courtyard_horizontal_width_m (Two horizontal segments (have two, always with same size) width)
- courtyard_vertical_length_m (Two vertical segments (have two, always with same size) length)
- courtyard_vertical_width_m (Two vertical segments (have two, always with same size) width)
Important: Length always means a horizontal (X-direction) measurement.
Width always means a vertical (Y-direction) measurement.
These labels do not depend on which side is visually longer or shorter.
"""

# =========================
# main loop
# =========================
def main():
    # RUN_MODE loading
    if RUN_MODE not in MODE_CONFIGS:
        raise ValueError(f"Invalid RUN_MODE: {RUN_MODE}. Choose from: {list(MODE_CONFIGS.keys())}")
    
    mode_cfg = MODE_CONFIGS[RUN_MODE]
    
    # strategies
    STRATEGY_CONFIG["self_consistency"]["enabled"] = mode_cfg["self_consistency"]
    STRATEGY_CONFIG["self_consistency"]["num_samples"] = mode_cfg["samples_per_model"]
    STRATEGY_CONFIG["self_consistency"]["temperature"] = mode_cfg["temperature"]
    STRATEGY_CONFIG["cross_consistency"]["enabled"] = mode_cfg["cross_consistency"]

    if RUN_MODE == "cross_consistency":
        active_providers = ["gemini", "openai", "qwen", "claude"]
    elif RUN_MODE == "self_consistency":
        active_providers = ["gemini", "openai", "qwen", "claude"]
    else:
        active_providers = [candidate]
        
        
    default_provider = active_providers[0]  # log and fallback, if the aggregation wrong, will fallback to the first one
    samples_per_model = mode_cfg["samples_per_model"]
    temperature = mode_cfg["temperature"]
    
    print(f"🚀 Run Mode: {RUN_MODE} - {mode_cfg['description']}")
    print(f"🤖 Active Providers: {active_providers}")
    print(f"📊 Samples per Model: {samples_per_model}, Temperature: {temperature}")
    # print(f"⚙️  Strategy Config: {json.dumps({k: v for k,v in STRATEGY_CONFIG.items() if v.get('enabled')}, indent=2)}\n")
    
    # input
    image_dir = Path(picture_path)
    image_exts = [".png", ".jpg", ".jpeg", ".webp"]
    
    image_files = []
    for ext in image_exts:
        image_files.extend(image_dir.glob(f"*{ext}"))
    image_files = sorted(image_files)
    
    print(f"📁 Found {len(image_files)} images in {image_dir}")
    if not image_files:
        print("⚠️  No images found. Please check the directory path.")
        return
    
    # result
    all_text_results = []
    
    for i, image_path in enumerate(image_files, 1):
        print(f"\n{'='*60}")
        print(f"🖼️  [{i}/{len(image_files)}] Processing: {image_path.name}")
        print(f"{'='*60}")
        
        try:
            candidate_texts = []
            candidate_parsed = []
            final_text = ""
            final_parsed = None

            # ── Mode A: baseline ──
            if RUN_MODE == "baseline":
                provider = active_providers[0]

                candidate_texts = []
                candidate_parsed = []

                for s_idx in range(samples_per_model):
                    print(f"  🎲 Baseline calling {provider}#{s_idx+1} (temp={temperature})...")

                    try:
                        text = call_model_single(
                            provider=provider,
                            image_path=image_path,
                            prompt=picture_system_prompt,
                            temperature=temperature,
                            stream=False
                        )

                        parsed = parse_geometry_output(text)

                        if parsed:
                            parsed["_source_model"] = provider
                            parsed["_sample_idx"] = s_idx
                            print(f"    ✓ Parsed: shape={parsed['shape']}, dims={len(parsed['dimensions'])}")
                            pprint.pprint(parsed["dimensions"])
                            candidate_parsed.append(parsed)
                        else:
                            print(f"    ⚠ Parse failed for {provider}#{s_idx+1}")

                        candidate_texts.append(text)

                    except Exception as e:
                        print(f"    ✗ Baseline {provider}#{s_idx+1} error: {e}")
                        continue

                # Only merge multiple baseline runs; no retry, no reflective verification
                final_text, final_parsed = finalize_from_candidates(
                    candidate_texts, candidate_parsed, user_text_prompt
                )
                
            # ── Mode B: self_consistency ──
            elif RUN_MODE == "self_consistency":
                provider_sequence = ["gemini", "openai", "qwen", "claude"]
                best_text = ""
                best_parsed = None
                best_conf = -1.0

                for provider in provider_sequence:
                    print(f"\n  🔁 Self-consistency trying provider: {provider}")

                    candidate_texts, candidate_parsed = run_single_provider_pipeline(
                        provider=provider,
                        image_path=image_path,
                        prompt=picture_system_prompt,
                        samples_per_model=samples_per_model,
                        temperature=temperature,
                        default_provider=provider
                    )

                    temp_text, temp_parsed = finalize_from_candidates(
                        candidate_texts, candidate_parsed, user_text_prompt
                    )

                    temp_conf = 0.0
                    if temp_parsed is not None:
                        temp_conf = temp_parsed.get("confidence", 1.0)

                    print(f"  📌 Provider {provider} final confidence: {temp_conf:.2%}")

                    if temp_conf > best_conf:
                        best_conf = temp_conf
                        best_text = temp_text
                        best_parsed = temp_parsed

                    # 90% threshold, switch providers
                    if temp_conf >= 0.90:
                        print(f"  ✅ Provider {provider} reached 90% confidence. Stop switching.")
                        final_text = temp_text
                        final_parsed = temp_parsed
                        break
                    else:
                        print(f"  ⚠ Provider {provider} confidence < 90%, switching to next provider...")
                
                # select the highest confidence
                if not final_text:
                    print(f"  ⚠ No provider reached 90% confidence. Using best provider result ({best_conf:.2%}).")
                    final_text = best_text
                    final_parsed = best_parsed

            # ── Mode C: cross_consistency ──
            elif RUN_MODE == "cross_consistency":
                provider_sequence = ["gemini", "openai", "qwen", "claude"]

                all_candidate_texts = []
                all_candidate_parsed = []

                print(f"\n  🔁 Cross-consistency: running all providers together")

                for provider in provider_sequence:
                    print(f"\n  🤖 Running provider: {provider}")

                    provider_texts, provider_parsed = run_single_provider_pipeline(
                        provider=provider,
                        image_path=image_path,
                        prompt=picture_system_prompt,
                        samples_per_model=samples_per_model,
                        temperature=temperature,
                        default_provider=provider
                    )

                    print(f"  📌 Provider {provider} returned {len(provider_parsed)} valid parsed result(s)")

                    all_candidate_texts.extend(provider_texts)
                    all_candidate_parsed.extend(provider_parsed)

                print(
                    f"\n  📊 Cross-provider aggregation: "
                    f"{len(all_candidate_parsed)} valid parsed result(s) "
                    f"from {len(provider_sequence)} providers"
                )

                final_text, final_parsed = finalize_from_candidates(
                    all_candidate_texts,
                    all_candidate_parsed,
                    user_text_prompt
                )

                if final_parsed is not None:
                    print(
                        f"  ✅ Final aggregated confidence: "
                        f"{final_parsed.get('confidence', 0.0):.2%}"
                    )
                else:
                    print("  ⚠ Aggregation returned no parsed consensus, fallback text used.")

            else:
                raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")
                
            # save
            if final_text.strip():
                print(f"\n📋 Final Output Preview:")
                print(f"{'-'*40}")
                preview = final_text[:600] + ("..." if len(final_text)>600 else "")
                print(preview)
                print(f"{'-'*40}")
                all_text_results.append(final_text)
            
            print(f"✅ [{i}/{len(image_files)}] Done")
            
        except Exception as e:
            print(f"✗ Error processing {image_path.name}: {e}")
            import traceback
            traceback.print_exc()
            continue


    # =========================
    # save in python file
    # =========================
    DIMENSION_LABELS = {
        # Rectangle
        "rectangle_overall_length_m": "{a} Overall building length",
        "rectangle_overall_width_m": "{b} Overall building width",
        # L-shaped
        "l_horizontal_length_m": "{h_a} Horizontal segment length",
        "l_horizontal_width_m": "{h_b} Horizontal segment width",
        "l_vertical_length_m": "{v_a} Vertical segment length",
        "l_vertical_width_m": "{v_b} Vertical segment width",
        # U-shaped
        "u_main_length_m": "{h_a} Length of the horizontal segment",
        "u_main_width_m": "{h_b} Width of the horizontal segment",
        "u_left_length_m": "{v1_a} Length of the left vertical segment",
        "u_left_width_m": "{v1_b} Width of the left vertical segment.",
        "u_right_length_m": "{v2_a} Length of the right vertical segment",
        "u_right_width_m": "{v2_b} Width of the right vertical segment",
        # T-shaped
        "t_horizontal_length_m": "{h_a} Horizontal segment length",
        "t_horizontal_width_m": "{h_b} Horizontal segment width",
        "t_vertical_length_m": "{v_a} Vertical segment length",
        "t_vertical_width_m": "{v_b} Vertical segment width",
        "t_edge_offset_m": "{v_x} The distance from the vertical segment to the edge of the horizontal segment",
        # Courtyard
        "courtyard_horizontal_length_m": "{a} Two horizontal segments (have two, always with same size) length",
        "courtyard_horizontal_width_m": "{y} Two horizontal segments (have two, always with same size) width",
        "courtyard_vertical_length_m": "{x} Two vertical segments (have two, always with same size) length",
        "courtyard_vertical_width_m": "{b} Two vertical segments (have two, always with same size) width",
    }

    def apply_dimension_label_replacements(text: str, mapping: dict) -> str:

        if not text:
            return text

        items = sorted(mapping.items(), key=lambda kv: len(kv[0]), reverse=True)

        for k, label in items:
            text = re.sub(
                rf'(^|\n)(\s*)(?:[-•]\s*)?{re.escape(k)}\s*:',
                rf'\1\2{label}:',
                text
            )
        return text

    with open(output_py, "w", encoding="utf-8") as f:
        f.write(f"# Auto-generated prompts - Run Mode: {RUN_MODE}\n")
        f.write(f"# Strategies: {list(k for k, v in STRATEGY_CONFIG.items() if v.get('enabled'))}\n\n")
        f.write("DIMENSION_LABELS = ")
        f.write(pprint.pformat(DIMENSION_LABELS, width=120))
        f.write("\n\n")

        for idx, text in enumerate(all_text_results, 1):
            replaced = apply_dimension_label_replacements(text, DIMENSION_LABELS)
            safe_text = replaced.replace('"""', "'''")
            f.write(f'user_prompt_{idx} = """{safe_text}"""\n\n')

        f.write("USER_PROMPTS = {\n")
        f.write("    k: v for k, v in globals().items()\n")
        f.write('    if k.startswith("user_prompt_")\n')
        f.write("}\n")

    print(f"\n🎉 All {len(all_text_results)} prompts saved to `{output_py}`")

if __name__ == "__main__":
    main()

🚀 Run Mode: baseline - Single Model + Multi Sample Baseline
🤖 Active Providers: ['gemini']
📊 Samples per Model: 3, Temperature: 0
📁 Found 50 images in Standard_Test

🖼️  [1/50] Processing: L_0.png
  🎲 Baseline calling gemini#1 (temp=0)...
    ✓ Parsed: shape=L-shaped, dims=4
{'l_horizontal_length_m': 99.07,
 'l_horizontal_width_m': 22.64,
 'l_vertical_length_m': 23.44,
 'l_vertical_width_m': 45.87}
  🎲 Baseline calling gemini#2 (temp=0)...
    ✓ Parsed: shape=L-shaped, dims=4
{'l_horizontal_length_m': 99.07,
 'l_horizontal_width_m': 22.64,
 'l_vertical_length_m': 23.44,
 'l_vertical_width_m': 45.87}
  🎲 Baseline calling gemini#3 (temp=0)...
    ✓ Parsed: shape=L-shaped, dims=4
{'l_horizontal_length_m': 99.07,
 'l_horizontal_width_m': 22.64,
 'l_vertical_length_m': 23.44,
 'l_vertical_width_m': 68.51}
  📊 Aggregating 3 valid results from 1 model(s)...
  📋 Aggregation Params: shape=L-shaped, confidence=91.67%, total_source_count=3, shape_consensus_source_count=3

📋 Final Output Preview:
